# 00 — Setup (CPU runtime)

**Runtime > Change runtime type > CPU** (no GPU needed).

This notebook mounts Google Drive, clones or updates the GitHub repo, installs CMake/g++/OpenMP and Python plotting packages, writes machine info to Drive, and checks the environment.

Edit only `REPO_URL` below.

In [ ]:
# THE ONLY CELL YOU SHOULD EDIT
REPO_URL = "https://github.com/ShubhamDudhbhate-VJTI/Scientific_Calculator.git"  # public HTTPS clone URL
DRIVE_ROOT = "/content/drive/MyDrive/parallel-sci-calc"
CLONE_DIR = "/content/parallel-sci-calc"
print("REPO_URL =", REPO_URL)
print("DRIVE_ROOT =", DRIVE_ROOT)
with open("/tmp/psc_env.sh", "w") as f:
    f.write(f"export REPO_URL='{REPO_URL}'\n")
    f.write(f"export DRIVE_ROOT='{DRIVE_ROOT}'\n")
    f.write(f"export CLONE_DIR='{CLONE_DIR}'\n")
print("OK: wrote /tmp/psc_env.sh")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("OK: Drive mounted")

In [ ]:
%%bash
source /tmp/psc_env.sh
mkdir -p "${DRIVE_ROOT}/results" "${DRIVE_ROOT}/logs"
echo "OK: Drive directories ${DRIVE_ROOT}/results"

In [ ]:
%%bash
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq
apt-get install -y -qq cmake g++ libomp-dev git
echo "OK: apt packages cmake g++ libomp-dev git"

In [ ]:
%%bash
pip install -q matplotlib pandas
echo "OK: pip matplotlib pandas"

In [ ]:
%%bash
source /tmp/psc_env.sh
set -euo pipefail
if [[ "${REPO_URL}" == *YOUR_USER* ]]; then
  echo "FAIL: set REPO_URL to your real GitHub clone URL"
  exit 1
fi
if [[ -d "${CLONE_DIR}/.git" ]]; then
  git -C "${CLONE_DIR}" pull --ff-only
  echo "OK: git pull ${CLONE_DIR}"
else
  rm -rf "${CLONE_DIR}"
  git clone "${REPO_URL}" "${CLONE_DIR}"
  echo "OK: git clone ${CLONE_DIR}"
fi
ls "${CLONE_DIR}/CMakeLists.txt"
chmod +x "${CLONE_DIR}/scripts/"*.sh "${CLONE_DIR}/bench/"*.sh 2>/dev/null || true

In [ ]:
%%bash
source /tmp/psc_env.sh
bash "${CLONE_DIR}/scripts/colab_machine_info.sh" "${DRIVE_ROOT}/results/machine_info.txt"
echo "OK: machine_info.txt"

In [ ]:
# Environment check — OK or a clear error per item
import os
import shutil

checks = {
    "Drive mounted": os.path.isdir("/content/drive/MyDrive"),
    "repo present": os.path.isfile("/content/parallel-sci-calc/CMakeLists.txt"),
    "cmake": shutil.which("cmake") is not None,
    "g++": shutil.which("g++") is not None,
}
failed = False
for name, ok in checks.items():
    print(("OK  " if ok else "FAIL"), name)
    failed = failed or (not ok)

import subprocess, tempfile, textwrap
src = textwrap.dedent(r'''
#include <omp.h>
#include <cstdio>
int main() {
  std::printf("OpenMP max threads = %d\n", omp_get_max_threads());
  return 0;
}
''')
omp_ok = False
try:
    path = "/tmp/omp_probe.cpp"
    with open(path, "w") as f:
        f.write(src)
    subprocess.check_call(["g++", "-fopenmp", path, "-o", "/tmp/omp_probe"])
    out = subprocess.check_output(["/tmp/omp_probe"], text=True)
    print("OK  OpenMP compile/run:", out.strip())
    omp_ok = True
except Exception as exc:
    print("FAIL OpenMP:", exc)
    failed = True

if failed:
    raise SystemExit("Environment check failed. Fix the FAIL lines before notebook 01.")
print("OK: all environment checks passed")